In [20]:
# Parámetros que serán recibidos desde el pipeline de Fabric
source_schema = "QAD"
source_table  = "EMP_MSTR"

target_layer  = "lh_silver_erp"
target_schema = "ecp"
target_table  = "employees"

# Nombre de la tabla de control
control_table = "bronze_to_silver_control"

StatementMeta(, 72507493-32bc-448c-82b3-60026cc8ffaa, 11, Finished, Available, Finished)

In [21]:
# Importaciones y Configuración 
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import udf, expr, col
from pyspark.sql.types import StringType
from pyspark.sql import functions as F, types as T

import json, notebookutils
from delta.tables import DeltaTable

import datetime  

# Lista de preposiciones y artículos a mantener en minúsculas
LOWERCASE_WORDS = {'a', 'ante', 'bajo', 'cabe', 'con', 'contra', 'de', 'desde', 'en', 
                   'entre', 'hacia', 'hasta', 'para', 'por', 'según', 'sin', 'so', 
                   'sobre', 'tras', 'y', 'e', 'ni', 'que', 'el', 'la', 'los', 'las', 'un', 'una', 
                   'S.' 'SA', 'S.A.', 'S.A', 'CV', 'C.V.','C.V','MAB','ZDI','LLC','USA','R.L.'}


# Configuración para LEER fechas antiguas de fuentes Parquet/Delta
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")

# Configuración para ESCRIBIR fechas antiguas a destinos Parquet/Delta (RECOMENDADA)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

StatementMeta(, 72507493-32bc-448c-82b3-60026cc8ffaa, 12, Finished, Available, Finished)

In [22]:
# --- Función de Logging ---
def update_task_status(status, message):
    """Actualiza la tabla de control con el estado final de la ejecución."""
    try:
        safe_message = message.replace("'", "''")
        update_query = f"""
            UPDATE lh_control_erp.dbo.{control_table}
            SET 
                last_run_status = '{status}', 
                last_message = '{safe_message}',
                last_run_at = current_timestamp(),
                last_success_run_at = {'current_timestamp()' if status == 'Success' else 'last_success_run_at'}
            WHERE 
                target_layer = '{target_layer}' AND 
                target_schema = '{target_schema}' AND 
                target_table = '{target_table}' AND
                is_active = 1
        """

        spark.sql(update_query)
        print(f"Log updated: Status='{status}', Message='{message}'")
    except Exception as e:
        print(f"FATAL: Could not update control table log. Reason: {e}")


def to_custom_title_case(text):
    """
    Convierte un string a tipo título, excepto ciertas preposiciones y artículos.
    La primera palabra siempre se capitaliza.
    """
    if text is None:
        return None
    
    words = text.lower().split()
    title_cased_words = [word.capitalize() if word not in LOWERCASE_WORDS else word for word in words]
    
    if title_cased_words:
        title_cased_words[0] = title_cased_words[0].capitalize()
        
    return " ".join(title_cased_words)

# Registra la función como una UDF de Spark para poder usarla en transformaciones
custom_title_case_udf = udf(to_custom_title_case, StringType())

def apply_company_code_fix(
    df,
    col_name="company_code",
    mapping=None,
    normalize=True,
    coerce_non_string=False,
    verbose=True
):
    """
    Normaliza company_code:
      ST -> STU
      PS -> PRO
    - Idempotente: si no existe la columna, no hace nada.
    - Usa map lookup con sintaxis moderna: repl[key_expr]
    """
    # localizar la columna real (case-insensitive)
    real_col = next((c for c in df.columns if c.lower() == col_name.lower()), None)
    if real_col is None:
        if verbose:
            print(f"[apply_company_code_fix] Columna '{col_name}' no existe. Sin cambios.")
        return df

    # mapeo base (+ overrides opcionales)
    base_map = {"ST": "STU", "PS": "PRO"}
    if mapping:
        base_map.update({str(k).upper(): str(v) for k, v in mapping.items()})

    # construir el map dinamicamente
    kv = []
    for k, v in base_map.items():
        kv += [F.lit(k), F.lit(v)]
    repl = F.create_map(*kv)

    # tipo de dato de la columna
    dtype = next(f.dataType for f in df.schema.fields if f.name == real_col)
    is_string = isinstance(dtype, T.StringType)

    if is_string:
        key_expr = F.upper(F.trim(F.col(real_col))) if normalize else F.col(real_col)
        new_value = F.coalesce(repl[key_expr], F.col(real_col))   # << uso de índice []
        return df.withColumn(real_col, new_value)

    if coerce_non_string:
        key_expr = (F.upper(F.trim(F.col(real_col).cast("string")))
                    if normalize else F.col(real_col).cast("string"))
        new_value = F.coalesce(repl[key_expr], F.col(real_col).cast("string")).cast(dtype)
        return df.withColumn(real_col, new_value)

    if verbose:
        print(f"[apply_company_code_fix] '{real_col}' es {dtype}. Omitido (coerce_non_string=False).")
    return df

def validate_mapping_compatibility(tasks):
    """
    Compara los column_mapping_json de todas las tareas para asegurar que
    el esquema final de Silver sea idéntico entre ellas.
    """
    schemas_info = []
    
    for config in tasks:
        mapping = json.loads(config["column_mapping_json"])
        # Creamos un set de tuplas (nombre_columna_destino, tipo_dato)
        # Esto ignora el orden y se enfoca en que el resultado final sea compatible
        schema_signature = sorted([(m['target'], m['cast'].upper()) for m in mapping])
        schemas_info.append({
            "source": f"{config['source_layer']}.{config['source_table']}",
            "signature": schema_signature
        })
    
    # Comparamos todos contra el primero
    reference = schemas_info[0]
    errors = []
    
    for i in range(1, len(schemas_info)):
        current = schemas_info[i]
        if current["signature"] != reference["signature"]:
            # Identificar diferencias específicas para el log
            diff_ref = set(reference["signature"]) - set(current["signature"])
            diff_cur = set(current["signature"]) - set(reference["signature"])
            
            error_detail = f"Discrepancia entre {reference['source']} y {current['source']}. "
            if diff_ref: error_detail += f"Faltan o cambian en destino: {diff_ref}. "
            if diff_cur: error_detail += f"Nuevos o diferentes en origen: {diff_cur}."
            errors.append(error_detail)
            
    return errors

StatementMeta(, 72507493-32bc-448c-82b3-60026cc8ffaa, 13, Finished, Available, Finished)

In [23]:
print(f"\n--- Iniciando carga Full para: {source_table} -> {target_table} ---")
try:
    # 1. LEER METADATOS DE LA TAREA
    print(f"--- Fetching configurations for: {target_table} ---")
    config_df = spark.sql(f"""
        SELECT * FROM {control_table} 
        WHERE target_layer = '{target_layer}' 
        AND target_schema = '{target_schema}' 
        AND target_table = '{target_table}'
        AND is_active = 1
    """)

    if config_df.isEmpty():
        raise ValueError(f"Task for {target_table} not found or is disabled.")

    tasks = config_df.collect()

    if len(tasks) > 1:
        print("--- Validating schema compatibility between sources ---")
        schema_errors = validate_mapping_compatibility(tasks)
    
        if schema_errors:
            error_report = " | ".join(schema_errors)
            update_task_status('Failed', f"Schema Mismatch: {error_report}")
            raise ValueError(f"Incompatible schemas detected: {error_report}")
        else:
            print("✅ All source mappings are compatible.")

    # --- LISTA PARA ACUMULAR LOS DATAFRAMES ---
    all_bronze_transformed_dfs = []

    # 2. CICLO DE TRANSFORMACIÓN (SIN ESCRITURA)
    for config in tasks:
        task_id          = config["task_id"]
        source_layer     = config["source_layer"]
        source_schema    = config["source_schema"]
        source_table     = config["source_table"]
        mapping_json     = config["column_mapping_json"]
    
        print(f"   Extracting and mapping: {source_layer}.{source_schema}.{source_table}")

        try:
            # Leer el origen específico de esta iteración
            current_df = spark.table(f"{source_layer}.{source_schema}.{source_table}")
        
            # Aplicar el mapeo dinámico para que todas las iteraciones tengan el mismo esquema
            mappings = json.loads(mapping_json)
            select_expressions = []
        
            for m in mappings:
                col_expr = expr(m['source']).cast(m['cast'])
            
                if m.get('transform') == 'title_case':
                    final_expr = custom_title_case_udf(col_expr).alias(m['target'])
                else:
                    final_expr = col_expr.alias(m['target'])
            
                select_expressions.append(final_expr)

            # Crear el DF transformado y guardarlo en nuestra lista
            transformed_df = current_df.select(*select_expressions)
            all_bronze_transformed_dfs.append(transformed_df)
        
        except Exception as e:
            print(f"   ERROR procesando {source_table}: {e}")
            # Aquí decidimos: si un origen falla, ¿detenemos todo o seguimos con los demás?
            raise e 

    # 3. UNIÓN Y ESCRITURA FINAL (FUERA DEL CICLO)
    if all_bronze_transformed_dfs:
        print(f"\n--- Unifying all sources and performing Overwrite ---")
    
        # Unir todos los DataFrames acumulados en uno solo
        # unionByName es más seguro que union() porque empareja por nombre de columna
        final_silver_df = reduce(DataFrame.unionByName, all_bronze_transformed_dfs)
    
        # Definir ruta de destino
        target_table_full_name = f"{target_layer}.{target_schema}.{target_table}"
    
        # Escritura única tipo OVERWRITE
        (
            final_silver_df
            .write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table_full_name)
        )
    
        # Actualizar control como exitoso
        update_task_status('Success', f'Full overwrite successful. Consolidated {len(all_bronze_transformed_dfs)} sources.')
        print(f"--- Process finished successfully for {target_table_full_name} ---")
    else:
        print("No dataframes were collected. Nothing to write.")
        update_task_status('Success', f'No dataframes were collected. Nothing to write.')

except Exception as e:
    # **PASO 4: ACTUALIZAR CONTROL (Fallo)**
    error_message = str(e).replace('\n', ' ').replace('\r', '').replace("'", "''") 
    print(f"ERROR: An exception occurred: {error_message}")

    update_task_status('Failed', error_message)

    # Propaga el error para que el pipeline de Fabric falle y te notifique.
    raise e

print(f"\n--- Proceso finalizado exitosamente para {source_table}. ---")

StatementMeta(, 72507493-32bc-448c-82b3-60026cc8ffaa, 14, Finished, Available, Finished)


--- Iniciando carga Full para: EMP_MSTR -> employees ---
--- Fetching configurations for: employees ---


   Extracting and mapping: lh_bronze_qad2.QAD.EMP_MSTR



--- Unifying all sources and performing Overwrite ---


Log updated: Status='Success', Message='Full overwrite successful. Consolidated 1 sources.'
--- Process finished successfully for lh_silver_erp.ecp.employees ---

--- Proceso finalizado exitosamente para EMP_MSTR. ---
